# 🕰️ Notebook 1: Lamport Clocks — and why they're not enough

**The big question:** *"In a distributed system, what does it even mean to say event A happened before event B?"*

Wall-clock time on different machines drifts. You can't trust it. Leslie Lamport's insight (1978): we don't need real time — we just need a counter that respects **causality**.

A **Lamport clock** is a single integer per process. Rules:

1. Before any local event, increment your clock.
2. When sending a message, attach your current clock value.
3. When receiving a message with timestamp `t`, set your clock to `max(local, t) + 1`.

This guarantees: if A *causally happened before* B, then `clock(A) < clock(B)`.

But the converse is not true! `clock(A) < clock(B)` does **not** imply A came before B — they might be **concurrent** (caused by neither). Lamport clocks **lose** that information.

In notebook 2 we'll fix that with **vector clocks**.

## Learning objectives
- Implement a Lamport clock from scratch.
- See how it orders causally-related events.
- Find a case where it lies about concurrency.

In [ ]:
class LamportProcess:
    def __init__(self, name):
        self.name = name
        self.clock = 0
        self.history = []

    def local(self, label):
        self.clock += 1
        self.history.append((self.clock, f"{self.name}: {label}"))

    def send(self, target, label):
        self.clock += 1
        self.history.append((self.clock, f"{self.name}: send '{label}' -> {target.name}"))
        return self.clock, label

    def recv(self, sender_clock, label):
        self.clock = max(self.clock, sender_clock) + 1
        self.history.append((self.clock, f"{self.name}: recv '{label}'"))

A = LamportProcess("A")
B = LamportProcess("B")
C = LamportProcess("C")

A.local("write x=1")
m1 = A.send(B, "x=1")
B.recv(*m1)
B.local("write y=2")
m2 = B.send(C, "y=2")
C.recv(*m2)
C.local("write z=3")

# A and C also do some independent work nobody told each other about
A.local("indep A1")
C.local("indep C1")

for t, ev in sorted(A.history + B.history + C.history):
    print(f"clock={t:2d}  {ev}")

## 🤔 The trap

Look at the timeline. `A: indep A1` and `C: indep C1` are **concurrent** — neither caused the other. But Lamport clocks give them different timestamps. If you only see the timestamps, you'd guess one came before the other.

That's the limitation: Lamport gives you a *total order* (no two events have the same time), but the order it gives you can be misleading. You can't tell **causality** from **coincidence**.